# Day 1 — Data Collection & Structuring
**LTLab Fellowship | Nairobi House Price Prediction Sprint**

Scrapes 300–1500+ Nairobi property listings from **10 Kenyan real-estate platforms** and consolidates them into a single clean raw dataset.

| # | Source | Domain | Specialty |
|---|--------|---------|----------|
| 1 | BuyRentKenya | buyrentkenya.com | Leading Kenya portal |
| 2 | Property24 Kenya | property24.co.ke | Professional agent listings |
| 3 | PropertyPro Kenya | propertypro.co.ke | Residential & commercial |
| 4 | Maploti | maploti.com | Verified, investor-grade |
| 5 | PigiaMe | pigiame.co.ke | Classifieds, private sellers |
| 6 | Kenya Property Centre | kenyapropertycentre.com | Upscale & veteran portal |
| 7 | Jiji Kenya | jiji.co.ke | Largest classifieds platform |
| 8 | Commercial Property Kenya | commercialpropertykenya.com | Commercial + premium resi |
| 9 | Peponi Real Estate | peponirealestate.com | Boutique agency |
| 10 | Hauzisha | hauzisha.co.ke | Affordable & youth market |

**Output files:**
- `data/raw_listings.csv` — merged, deduplicated listings (sprint spec)
- `data/raw_listings_by_source.csv` — same data with `Source` column for auditing
- `data/data_dictionary.csv` — column definitions

## 1. Imports & Setup

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import os
from pathlib import Path

Path('data').mkdir(exist_ok=True)
print('Output directory ready: data/')

Output directory ready: data/


## 2. Configuration

In [2]:
# ── Global settings ────────────────────────────────────────────────────────────
REQUEST_DELAY         = 1.5
MAX_CONSECUTIVE_EMPTY = 4
REQUEST_TIMEOUT       = 20

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Connection': 'keep-alive',
}

# ── Source registry ────────────────────────────────────────────────────────────
# pagination:
#   query  ->  ?page=N  (page 1 = base_url, page 2+ appends ?page=N)
#   p24    ->  trailing segment /1 /2 /3 (Property24 style)
#   path   ->  /page/N/
#   jiji   ->  ?page=N&currency=KES
SOURCES = [
    {
        'name'      : 'BuyRentKenya',
        'base_url'  : 'https://www.buyrentkenya.com/property-for-sale/nairobi',
        'pagination': 'query',
        'scraper'   : 'buyrentkenya',
        'target'    : 300,
    },
    {
        'name'      : 'Property24',
        'base_url'  : 'https://www.property24.co.ke/property-for-sale/nairobi/1',
        'pagination': 'p24',
        'scraper'   : 'property24',
        'target'    : 250,
    },
    {
        'name'      : 'PropertyPro',
        'base_url'  : 'https://www.propertypro.co.ke/property-for-sale/in/nairobi',
        'pagination': 'query',
        'scraper'   : 'propertypro',
        'target'    : 200,
    },
    {
        'name'      : 'Maploti',
        'base_url'  : 'https://www.maploti.com/properties/for-sale/nairobi',
        'pagination': 'query',
        'scraper'   : 'maploti',
        'target'    : 150,
    },
    {
        'name'      : 'PigiaMe',
        'base_url'  : 'https://www.pigiame.co.ke/houses-apartments-for-sale/nairobi',
        'pagination': 'query',
        'scraper'   : 'pigiame',
        'target'    : 150,
    },
    {
        'name'      : 'KenyaPropertyCentre',
        'base_url'  : 'https://www.kenyapropertycentre.com/properties-for-sale/nairobi',
        'pagination': 'query',
        'scraper'   : 'kpc',
        'target'    : 150,
    },
    {
        'name'      : 'Jiji',
        'base_url'  : 'https://jiji.co.ke/nairobi/houses-apartments-for-sale',
        'pagination': 'jiji',
        'scraper'   : 'jiji',
        'target'    : 200,
    },
    {
        'name'      : 'CommercialPropertyKenya',
        'base_url'  : 'https://www.commercialpropertykenya.com/property-for-sale/nairobi',
        'pagination': 'query',
        'scraper'   : 'generic',
        'target'    : 100,
    },
    {
        'name'      : 'PeponiRealEstate',
        'base_url'  : 'https://www.peponirealestate.com/properties/?status=for-sale',
        'pagination': 'path',
        'scraper'   : 'generic',
        'target'    : 80,
    },
    {
        'name'      : 'Hauzisha',
        'base_url'  : 'https://hauzisha.co.ke/properties?type=sale&city=Nairobi',
        'pagination': 'query',
        'scraper'   : 'generic',
        'target'    : 100,
    },
]

# ── Known Nairobi sub-locations (sorted longest-first to avoid partial matches) ─
NAIROBI_AREAS = sorted([
    'Westlands', 'Kilimani', 'Kileleshwa', 'Lavington', 'Karen',
    'Runda', 'Muthaiga', 'Gigiri', 'Parklands', 'Spring Valley',
    'Lower Kabete', 'Ridgeways', 'Rosslyn', 'Ruaka', 'Ruiru',
    'Thika Road', 'Kasarani', 'Roysambu', 'Syokimau', 'Mlolongo',
    'Rongai', 'Ngong', 'Athi River', 'Kitengela', 'Embakasi',
    'South B', 'South C', 'Langata', 'Eastleigh', 'Huruma',
    'Githurai', 'Zimmerman', 'Imara Daima', 'Donholm', 'Buruburu',
    'Upper Hill', 'Upperhill', 'Adams Arcade', 'Dagoretti',
    'Kabete', 'Kikuyu', 'Limuru', 'Tigoni', 'Kitisuru',
    'Loresho', 'Brookside', 'Riverside', 'Nairobi CBD', 'Hurlingham',
    'Ngara', 'Pangani', 'Thome', 'Komarock', 'Mirema',
    'Mwiki', 'Kahawa', 'Umoja', 'Tena', 'Fedha',
    'Nairobi West', 'Ngumo', 'Madaraka', 'Garden Estate',
    'Membley', 'Juja', 'Kamiti', 'Mountain View', 'Peponi',
    'Highridge', 'Regen', 'Ngei', 'Ridgeway', 'Rehema',
    'Kahawa Wendani', 'Ruaraka', 'Clay City', 'Pipeline',
    'Buru Buru', 'Starehe', 'Mathare', 'Ziwani', 'Njiru',
    'Kayole', 'Dandora', 'Korogocho', 'Industrial Area',
    'Valley Arcade', 'Ngong Road', 'Yaya', 'Kilimani',
    'Kiambu', 'Thika', 'Ongata Rongai', 'South C',
], key=len, reverse=True)

# ── Amenity keywords ────────────────────────────────────────────────────────────
AMENITIES_LIST = [
    'Swimming Pool', 'Gym', 'Garden', 'Parking', 'Security',
    'Borehole', 'Lift', 'Elevator', 'En Suite', 'Balcony',
    'CCTV', 'Generator', 'Solar', 'Fibre', 'Wi-Fi',
    'Servant Quarter', 'SQ', 'Clubhouse', 'Playground', 'Gate',
    'Intercom', 'Backup Water', 'Rooftop', 'Concierge', 'Sauna',
    'Jacuzzi', 'Store', 'Laundry', 'Gazebo', 'Electric Fence',
]

print(f'Configuration loaded. {len(SOURCES)} sources registered:')
total_target = sum(s['target'] for s in SOURCES)
for s in SOURCES:
    print(f"  {s['name']:<30}  target={s['target']:>4}  scraper={s['scraper']}")
print(f'Total combined target: {total_target} listings')

Configuration loaded. 10 sources registered:
  BuyRentKenya                    target= 300  scraper=buyrentkenya
  Property24                      target= 250  scraper=property24
  PropertyPro                     target= 200  scraper=propertypro
  Maploti                         target= 150  scraper=maploti
  PigiaMe                         target= 150  scraper=pigiame
  KenyaPropertyCentre             target= 150  scraper=kpc
  Jiji                            target= 200  scraper=jiji
  CommercialPropertyKenya         target= 100  scraper=generic
  PeponiRealEstate                target=  80  scraper=generic
  Hauzisha                        target= 100  scraper=generic
Total combined target: 1680 listings


## 3. Shared Parsing Utilities

In [3]:
def parse_price(text):
    """Extract numeric KES price. Handles: KSh/KES prefix, xM shorthand, bare 6+ digit numbers."""
    if not text:
        return 'N/A'
    m = re.search(r'([\d]+(?:\.\d+)?)\s*[Mm](?:illion)?', text)
    if m:
        return str(int(float(m.group(1)) * 1_000_000))
    m = re.search(r'(?:KSh|KES|Kshs?)[\s,]?([\d][\d,\.]+)', text, re.IGNORECASE)
    if m:
        return re.sub(r'[^\d]', '', m.group(1))
    m = re.search(r'\b(\d[\d,]{5,})\b', text)
    if m:
        return re.sub(r'[^\d]', '', m.group(1))
    return 'N/A'


def parse_location(text, title=''):
    """Return the first recognised Nairobi sub-location (longest-match-first)."""
    combined = (text or '') + ' ' + (title or '')
    for area in NAIROBI_AREAS:
        if re.search(r'\b' + re.escape(area) + r'\b', combined, re.IGNORECASE):
            return area
    return 'Nairobi'


def parse_size(text):
    """Extract size in m². Converts sqft (x0.0929) and acres (x4046.86)."""
    if not text:
        return 'N/A'
    m = re.search(r'([\d,]+(?:\.\d+)?)\s*(?:m\u00b2|m\u00c2\u00b2|m\.?2|sqm|sq\.?m)',
                  text, re.IGNORECASE)
    if m:
        return re.sub(r'[^\d.]', '', m.group(1))
    m = re.search(r'([\d,]+(?:\.\d+)?)\s*(?:sq\.?\s*ft|sqft)', text, re.IGNORECASE)
    if m:
        return str(round(float(re.sub(r'[^\d.]', '', m.group(1))) * 0.0929, 1))
    m = re.search(r'([\d,]+(?:\.\d+)?)\s*acres?', text, re.IGNORECASE)
    if m:
        return str(round(float(re.sub(r'[^\d.]', '', m.group(1))) * 4046.86, 1))
    return 'N/A'


def parse_property_type(title, body=''):
    """Classify property type (first keyword match wins)."""
    combined = (title + ' ' + body).lower()
    for keywords, label in [
        (['townhouse', 'town house'],                 'Townhouse'),
        (['maisonette'],                              'Maisonette'),
        (['villa'],                                   'Villa'),
        (['bungalow'],                                'Bungalow'),
        (['studio'],                                  'Studio'),
        (['bedsitter', 'bedsit', 'bed-sitter'],       'Bedsitter'),
        (['commercial', 'office', 'shop',
          'godown', 'warehouse', 'retail'],           'Commercial'),
        (['land', 'plot', 'acre', 'acreage'],         'Land'),
        (['house', 'home', 'residence'],              'House'),
        (['apartment', 'flat', 'unit'],               'Apartment'),
    ]:
        if any(k in combined for k in keywords):
            return label
    return 'Apartment'


def parse_amenities(text):
    """Return comma-separated amenities found in text, or 'None'."""
    found = [am for am in AMENITIES_LIST if am.lower() in text.lower()]
    if 'Elevator' in found:
        found.remove('Elevator')
        if 'Lift' not in found:
            found.append('Lift')
    return ', '.join(found) if found else 'None'


def parse_int_field(text, *patterns):
    """Return group(1) of first matching pattern, else 'N/A'."""
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(1)
    return 'N/A'


def build_record(title, location, prop_type, beds, baths,
                 size, amenities, price, date, source):
    return {
        '_title'        : title,
        'Location'      : location,
        'Property Type' : prop_type,
        'Bedrooms'      : beds,
        'Bathrooms'     : baths,
        'Size (m2)'     : size,
        'Amenities'     : amenities,
        'Price (KES)'   : price,
        'Listing Date'  : date,
        'Source'        : source,
    }


print('Shared utilities ready.')

Shared utilities ready.


## 4. Site-Specific Page Parsers

In [4]:
# ══════════════════════════════════════════════════════════════════════
# PARSER A  —  BuyRentKenya  (buyrentkenya.com)
# ══════════════════════════════════════════════════════════════════════
def parse_buyrentkenya(soup, source):
    results = []
    cards = soup.find_all('article')
    if not cards:
        seen = set()
        for h3 in soup.find_all('h3'):
            c = h3.find_parent('div', class_=re.compile(r'relative|card|listing', re.I))
            if c and id(c) not in seen:
                seen.add(id(c))
                cards.append(c)
    for card in cards:
        try:
            h3 = card.find('h3')
            if not h3:
                continue
            title = h3.get_text(strip=True)
            if not title or len(title) < 5:
                continue
            body = card.get_text(separator=' ')
            pe = card.find(['p', 'span'], class_=re.compile(r'text-xl|font-bold|price', re.I))
            price = parse_price(pe.get_text() if pe else body)
            location = 'Nairobi'
            for p in card.find_all('p'):
                t = p.get_text(strip=True)
                if t and len(t) < 70 and 'KSh' not in t and 'Bedroom' not in t:
                    loc = parse_location(t, title)
                    if loc != 'Nairobi':
                        location = loc
                        break
            if location == 'Nairobi':
                location = parse_location(body, title)
            de = card.find(['p', 'span', 'time'], class_=re.compile(r'text-xs|date|time|listed', re.I))
            results.append(build_record(
                title=title, location=location,
                prop_type=parse_property_type(title, body),
                beds=parse_int_field(body, r'(\d+)\s*Bed(?:room)?s?', r'(\d+)\s*BR'),
                baths=parse_int_field(body, r'(\d+)\s*Bath(?:room)?s?', r'(\d+)\s*BA'),
                size=parse_size(body), amenities=parse_amenities(body),
                price=price, date=de.get_text(strip=True) if de else 'N/A', source=source,
            ))
        except Exception:
            continue
    return results

In [5]:
# ══════════════════════════════════════════════════════════════════════
# PARSER B  —  Property24 Kenya  (property24.co.ke)
# ══════════════════════════════════════════════════════════════════════
def parse_property24(soup, source):
    results = []
    cards = (
        soup.find_all('div', class_=re.compile(r'p24_regularTile|p24_listing|listingResult', re.I))
        or soup.find_all('div', attrs={'data-listing-id': True})
    )
    if not cards:
        return parse_generic(soup, source)
    for card in cards:
        try:
            h = card.find(['h2', 'h3', 'h4'])
            title = h.get_text(strip=True) if h else ''
            if not title or len(title) < 5:
                continue
            body = card.get_text(separator=' ')
            pe = card.find(['span', 'p'], class_=re.compile(r'price|amount', re.I))
            price = parse_price(pe.get_text() if pe else body)
            ae = card.find(['span', 'p', 'div'], class_=re.compile(r'address|location|suburb', re.I))
            location = parse_location(ae.get_text() if ae else body, title)
            de = card.find(['span', 'p'], class_=re.compile(r'date|listed|time', re.I))
            results.append(build_record(
                title=title, location=location,
                prop_type=parse_property_type(title, body),
                beds=parse_int_field(body, r'(\d+)\s*Bed(?:room)?s?', r'(\d+)\s*BR'),
                baths=parse_int_field(body, r'(\d+)\s*Bath(?:room)?s?', r'(\d+)\s*BA'),
                size=parse_size(body), amenities=parse_amenities(body),
                price=price, date=de.get_text(strip=True) if de else 'N/A', source=source,
            ))
        except Exception:
            continue
    return results

In [6]:
# ══════════════════════════════════════════════════════════════════════
# PARSER C  —  PropertyPro Kenya  (propertypro.co.ke)
# ══════════════════════════════════════════════════════════════════════
def parse_propertypro(soup, source):
    results = []
    cards = [
        c for c in soup.find_all('div', class_=re.compile(
            r'single.room.card|listings.card|property.card|col-md-6', re.I))
        if c.find(['h3', 'h4', 'h2'])
    ]
    if not cards:
        return parse_generic(soup, source)
    for card in cards:
        try:
            h = card.find(['h3', 'h4', 'h2'])
            title = h.get_text(strip=True) if h else ''
            if not title or len(title) < 5:
                continue
            body = card.get_text(separator=' ')
            pe = card.find(class_=re.compile(r'price|cost|amount', re.I))
            price = parse_price(pe.get_text() if pe else body)
            le = card.find(class_=re.compile(r'location|address|area', re.I))
            location = parse_location(le.get_text() if le else body, title)
            results.append(build_record(
                title=title, location=location,
                prop_type=parse_property_type(title, body),
                beds=parse_int_field(body, r'(\d+)\s*Bed(?:room)?s?', r'(\d+)\s*BR'),
                baths=parse_int_field(body, r'(\d+)\s*Bath(?:room)?s?', r'(\d+)\s*BA'),
                size=parse_size(body), amenities=parse_amenities(body),
                price=price, date='N/A', source=source,
            ))
        except Exception:
            continue
    return results

In [7]:
# ══════════════════════════════════════════════════════════════════════
# PARSER D  —  Maploti  (maploti.com)
# Modern Next.js portal — data-testid or class-based cards
# ══════════════════════════════════════════════════════════════════════
def parse_maploti(soup, source):
    results = []
    cards = (
        soup.find_all('div', attrs={'data-testid': re.compile(r'listing|property|card', re.I)})
        or soup.find_all('div', class_=re.compile(
            r'property[-_]?card|listing[-_]?card|result[-_]?item', re.I))
        or soup.find_all('article')
    )
    if not cards:
        return parse_generic(soup, source)
    for card in cards:
        try:
            h = card.find(['h2', 'h3', 'h4'])
            title = h.get_text(strip=True) if h else ''
            if not title or len(title) < 5:
                continue
            body = card.get_text(separator=' ')
            pe = card.find(class_=re.compile(r'price|amount|cost', re.I))
            price = parse_price(pe.get_text() if pe else body)
            le = card.find(class_=re.compile(r'location|address|neighbourhood|area', re.I))
            location = parse_location(le.get_text() if le else body, title)
            results.append(build_record(
                title=title, location=location,
                prop_type=parse_property_type(title, body),
                beds=parse_int_field(body, r'(\d+)\s*Bed(?:room)?s?', r'(\d+)\s*BR'),
                baths=parse_int_field(body, r'(\d+)\s*Bath(?:room)?s?', r'(\d+)\s*BA'),
                size=parse_size(body), amenities=parse_amenities(body),
                price=price, date='N/A', source=source,
            ))
        except Exception:
            continue
    return results

In [8]:
# ══════════════════════════════════════════════════════════════════════
# PARSER E  —  PigiaMe  (pigiame.co.ke)
# Large classifieds — listing-card or article tags
# ══════════════════════════════════════════════════════════════════════
def parse_pigiame(soup, source):
    results = []
    cards = [
        c for c in soup.find_all(
            ['article', 'div'],
            class_=re.compile(r'listing[-_]?card|ad[-_]?card|item[-_]?card|col-sm', re.I)
        ) if c.find(['h2', 'h3', 'h4'])
    ]
    if not cards:
        return parse_generic(soup, source)
    for card in cards:
        try:
            h = card.find(['h2', 'h3', 'h4'])
            title = h.get_text(strip=True) if h else ''
            if not title or len(title) < 5:
                continue
            body = card.get_text(separator=' ')
            pe = card.find(class_=re.compile(r'price|amount|currency', re.I))
            price = parse_price(pe.get_text() if pe else body)
            le = card.find(class_=re.compile(r'location|address|city|region', re.I))
            location = parse_location(le.get_text() if le else body, title)
            de = card.find(class_=re.compile(r'date|time|posted|ago', re.I))
            results.append(build_record(
                title=title, location=location,
                prop_type=parse_property_type(title, body),
                beds=parse_int_field(body, r'(\d+)\s*Bed(?:room)?s?', r'(\d+)\s*BR'),
                baths=parse_int_field(body, r'(\d+)\s*Bath(?:room)?s?', r'(\d+)\s*BA'),
                size=parse_size(body), amenities=parse_amenities(body),
                price=price, date=de.get_text(strip=True) if de else 'N/A', source=source,
            ))
        except Exception:
            continue
    return results

In [9]:
# ══════════════════════════════════════════════════════════════════════
# PARSER F  —  Kenya Property Centre  (kenyapropertycentre.com)
# ══════════════════════════════════════════════════════════════════════
def parse_kpc(soup, source):
    results = []
    all_cards = soup.find_all(
        ['div', 'article'],
        class_=re.compile(r'listing|property|item|featured|card|col-md', re.I)
    )
    cards = [
        c for c in all_cards
        if c.find(['h2', 'h3', 'h4']) and len(list(c.parents)) < 14
        and (c.find(string=re.compile(r'KSh|KES|Ksh'))
             or c.find(class_=re.compile(r'price', re.I)))
    ]
    if not cards:
        return parse_generic(soup, source)
    seen = set()
    for card in cards:
        try:
            h = card.find(['h2', 'h3', 'h4'])
            title = h.get_text(strip=True) if h else ''
            if not title or len(title) < 5 or title in seen:
                continue
            seen.add(title)
            body = card.get_text(separator=' ')
            pe = card.find(class_=re.compile(r'price|amount|cost', re.I))
            price = parse_price(pe.get_text() if pe else body)
            le = card.find(class_=re.compile(r'location|address|area', re.I))
            location = parse_location(le.get_text() if le else body, title)
            results.append(build_record(
                title=title, location=location,
                prop_type=parse_property_type(title, body),
                beds=parse_int_field(body, r'(\d+)\s*Bed(?:room)?s?', r'(\d+)\s*BR'),
                baths=parse_int_field(body, r'(\d+)\s*Bath(?:room)?s?', r'(\d+)\s*BA'),
                size=parse_size(body), amenities=parse_amenities(body),
                price=price, date='N/A', source=source,
            ))
        except Exception:
            continue
    return results

In [10]:
# ══════════════════════════════════════════════════════════════════════
# PARSER G  —  Jiji Kenya  (jiji.co.ke)
# Kenya's largest classifieds — b-list-advert / b-advert-grid cards
# ══════════════════════════════════════════════════════════════════════
def parse_jiji(soup, source):
    results = []
    cards = (
        soup.find_all('div', class_=re.compile(r'b-list-advert|qa-advert', re.I))
        or soup.find_all('article', class_=re.compile(r'b-advert|advert', re.I))
        or soup.find_all('div', class_=re.compile(r'listing|property', re.I))
    )
    if not cards:
        return parse_generic(soup, source)
    for card in cards:
        try:
            h = card.find(['h2', 'h3', 'h4', 'a'])
            title = h.get_text(strip=True) if h else ''
            if not title or len(title) < 5:
                continue
            body = card.get_text(separator=' ')
            pe = card.find(['span', 'div', 'p'], class_=re.compile(r'price|amount', re.I))
            price = parse_price(pe.get_text() if pe else body)
            le = card.find(['span', 'div', 'p'], class_=re.compile(r'location|address|region|city', re.I))
            location = parse_location(le.get_text() if le else body, title)
            de = card.find(['span', 'time'], class_=re.compile(r'date|time|posted|ago|b-advert-date', re.I))
            results.append(build_record(
                title=title, location=location,
                prop_type=parse_property_type(title, body),
                beds=parse_int_field(body, r'(\d+)\s*Bed(?:room)?s?', r'(\d+)\s*BR'),
                baths=parse_int_field(body, r'(\d+)\s*Bath(?:room)?s?', r'(\d+)\s*BA'),
                size=parse_size(body), amenities=parse_amenities(body),
                price=price, date=de.get_text(strip=True) if de else 'N/A', source=source,
            ))
        except Exception:
            continue
    return results

In [11]:
# ══════════════════════════════════════════════════════════════════════
# PARSER H  —  Generic Fallback
# Used for: CommercialPropertyKenya, PeponiRealEstate, Hauzisha, and any
# site whose dedicated parser returns 0 results.
# Climbs from every heading to its nearest price-containing ancestor card.
# ══════════════════════════════════════════════════════════════════════
def parse_generic(soup, source):
    results = []
    seen_ids = set()
    for h_tag in soup.find_all(['h2', 'h3', 'h4']):
        title = h_tag.get_text(strip=True)
        if not title or len(title) < 5:
            continue
        card = None
        for ancestor in h_tag.parents:
            if ancestor.name in ('div', 'article', 'li', 'section'):
                if re.search(r'KSh|KES|Ksh|\d{6,}', ancestor.get_text()):
                    card = ancestor
                    break
        if card is None or id(card) in seen_ids:
            continue
        seen_ids.add(id(card))
        try:
            body = card.get_text(separator=' ')
            results.append(build_record(
                title=title, location=parse_location(body, title),
                prop_type=parse_property_type(title, body),
                beds=parse_int_field(body, r'(\d+)\s*Bed(?:room)?s?', r'(\d+)\s*BR'),
                baths=parse_int_field(body, r'(\d+)\s*Bath(?:room)?s?', r'(\d+)\s*BA'),
                size=parse_size(body), amenities=parse_amenities(body),
                price=parse_price(body), date='N/A', source=source,
            ))
        except Exception:
            continue
    return results


# ── Dispatch map ───────────────────────────────────────────────────────
PARSER_MAP = {
    'buyrentkenya' : parse_buyrentkenya,
    'property24'   : parse_property24,
    'propertypro'  : parse_propertypro,
    'maploti'      : parse_maploti,
    'pigiame'      : parse_pigiame,
    'kpc'          : parse_kpc,
    'jiji'         : parse_jiji,
    'generic'      : parse_generic,
}
print('All parsers registered:', list(PARSER_MAP.keys()))

All parsers registered: ['buyrentkenya', 'property24', 'propertypro', 'maploti', 'pigiame', 'kpc', 'jiji', 'generic']


## 5. Pagination Engine

In [12]:
def build_page_url(base_url, page, style):
    """
    Build the URL for page N given the site's pagination style.
      query  ->  base_url?page=N
      p24    ->  replace trailing /1 with /N
      path   ->  base_url/page/N/
      jiji   ->  base_url?page=N&currency=KES
    """
    if page == 1:
        return base_url
    if style in ('query', 'offset'):
        sep = '&' if '?' in base_url else '?'
        return f'{base_url}{sep}page={page}'
    if style == 'p24':
        return re.sub(r'/(\d+)/?$', f'/{page}/', base_url.rstrip('/') + '/')
    if style == 'path':
        return f'{base_url.rstrip("/")}/page/{page}/'
    if style == 'jiji':
        sep = '&' if '?' in base_url else '?'
        return f'{base_url}{sep}page={page}&currency=KES'
    return base_url


def scrape_source(cfg, session):
    """Paginate one source and return all raw listing dicts."""
    name      = cfg['name']
    base_url  = cfg['base_url']
    style     = cfg['pagination']
    parser_fn = PARSER_MAP[cfg['scraper']]
    target    = cfg['target']

    all_listings      = []
    page              = 1
    consecutive_empty = 0

    print(f'\n── {name}  (target: {target}) ──')

    while len(all_listings) < target * 1.5 and consecutive_empty < MAX_CONSECUTIVE_EMPTY:
        url = build_page_url(base_url, page, style)
        print(f'  Page {page:3d} | {len(all_listings):4d} so far', end='')

        try:
            resp = session.get(url, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 404:
                print('  [404 – end of results]')
                break
            if resp.status_code != 200:
                print(f'  [HTTP {resp.status_code}]')
                consecutive_empty += 1
                page += 1
                time.sleep(REQUEST_DELAY)
                continue
            soup  = BeautifulSoup(resp.content, 'html.parser')
            found = parser_fn(soup, name)
            if not found:
                print('  [0 listings]')
                consecutive_empty += 1
            else:
                all_listings.extend(found)
                consecutive_empty = 0
                print(f'  [{len(found)} listings]')
        except requests.exceptions.Timeout:
            print('  [timeout]')
            consecutive_empty += 1
        except Exception as exc:
            print(f'  [error: {exc}]')
            break

        page += 1
        time.sleep(REQUEST_DELAY)

    print(f'  -> {len(all_listings)} raw listings from {name}')
    return all_listings


print('Pagination engine ready.')

Pagination engine ready.


## 6. Run All 10 Sources

In [13]:
session = requests.Session()
session.headers.update(HEADERS)

all_raw = []

print('=' * 65)
print('  MULTI-SOURCE NAIROBI PROPERTY COLLECTION')
print(f'  Sources : {len(SOURCES)}')
print(f'  Combined target: {sum(s["target"] for s in SOURCES)} listings')
print('=' * 65)

for src in SOURCES:
    raw = scrape_source(src, session)
    all_raw.extend(raw)

print(f'\nTotal raw records collected: {len(all_raw)}')

  MULTI-SOURCE NAIROBI PROPERTY COLLECTION
  Sources : 10
  Combined target: 1680 listings

── BuyRentKenya  (target: 300) ──
  Page   1 |    0 so far  [27 listings]
  Page   2 |   27 so far  [27 listings]
  Page   3 |   54 so far  [27 listings]
  Page   4 |   81 so far  [27 listings]
  Page   5 |  108 so far  [27 listings]
  Page   6 |  135 so far  [27 listings]
  Page   7 |  162 so far  [27 listings]
  Page   8 |  189 so far  [27 listings]
  Page   9 |  216 so far  [27 listings]
  Page  10 |  243 so far  [27 listings]
  Page  11 |  270 so far  [27 listings]
  Page  12 |  297 so far  [27 listings]
  Page  13 |  324 so far  [27 listings]
  Page  14 |  351 so far  [27 listings]
  Page  15 |  378 so far  [27 listings]
  Page  16 |  405 so far  [27 listings]
  Page  17 |  432 so far  [27 listings]
  -> 459 raw listings from BuyRentKenya

── Property24  (target: 250) ──
  Page   1 |    0 so far  [404 – end of results]
  -> 0 raw listings from Property24

── PropertyPro  (target: 200) ──
  

## 7. Merge, Deduplicate & Finalise

In [14]:
df_raw = pd.DataFrame(all_raw)
print(f'Rows before dedup : {len(df_raw)}')

# Deduplicate on title + location + price (catches cross-site reposts)
df_raw = df_raw.drop_duplicates(subset=['_title', 'Location', 'Price (KES)'])
print(f'Rows after dedup  : {len(df_raw)}')

COL_ORDER = [
    'Location', 'Property Type', 'Bedrooms', 'Bathrooms',
    'Size (m2)', 'Amenities', 'Price (KES)', 'Listing Date'
]

df_listings    = df_raw[COL_ORDER].copy()
df_with_source = df_raw[COL_ORDER + ['Source']].copy()

print(f'\nFinal dataset : {df_listings.shape[0]} rows x {df_listings.shape[1]} columns')
print('\nPer-source breakdown (post-dedup):')
print(df_with_source.groupby('Source').size()
      .reset_index(name='Listings')
      .sort_values('Listings', ascending=False)
      .to_string(index=False))

Rows before dedup : 1495
Rows after dedup  : 926

Final dataset : 926 rows x 8 columns

Per-source breakdown (post-dedup):
                 Source  Listings
                   Jiji       283
            PropertyPro       222
    KenyaPropertyCentre       213
           BuyRentKenya       206
CommercialPropertyKenya         2


## 8. Data Quality Audit

In [15]:
print('── N/A audit ─────────────────────────────────────')
na_counts = (df_listings == 'N/A').sum()
na_pct    = (na_counts / len(df_listings) * 100).round(1)
print(pd.DataFrame({'N/A Count': na_counts, 'N/A %': na_pct}).to_string())
print(f'\nOverall N/A rate: {na_counts.sum() / df_listings.size * 100:.1f}%')

── N/A audit ─────────────────────────────────────
               N/A Count  N/A %
Location               0    0.0
Property Type          0    0.0
Bedrooms             134   14.5
Bathrooms            463   50.0
Size (m2)            460   49.7
Amenities              0    0.0
Price (KES)          228   24.6
Listing Date         912   98.5

Overall N/A rate: 29.7%


In [16]:
print('── Property type distribution ────────────────────')
print(df_listings['Property Type'].value_counts().to_string())

── Property type distribution ────────────────────
Property Type
Apartment     376
Land          170
Townhouse      96
House          75
Commercial     73
Villa          39
Maisonette     38
Studio         27
Bungalow       22
Bedsitter      10


In [17]:
print('── Top 20 locations ──────────────────────────────')
print(df_listings['Location'].value_counts().head(20).to_string())

── Top 20 locations ──────────────────────────────
Location
Westlands          159
Kilimani           129
Nairobi            127
Kileleshwa         101
Lavington           93
Karen               44
Kitisuru            20
Parklands           16
Valley Arcade       15
Runda               14
Kasarani            14
Hurlingham          13
Nairobi CBD         10
South C              9
Industrial Area      9
Roysambu             8
Dagoretti            8
Thika Road           8
Ngong Road           7
Embakasi             7


In [18]:
print('── Preview (first 10 rows) ───────────────────────')
df_listings.head(10)

── Preview (first 10 rows) ───────────────────────


,Location,Property Type,Bedrooms,Bathrooms,Size (m2),Amenities,Price (KES),Listing Date
0,Westlands,Land,1,2,65,Swimming Pool,5500000,N/A
1,Westlands,Villa,6,6,N/A,En Suite,260000000,N/A
2,Kileleshwa,Apartment,1,2,65,Swimming Pool,6000000,N/A
3,Kileleshwa,Apartment,1,1,47,Swimming Pool,5100000,N/A
4,Westlands,Land,N/A,N/A,N/A,None,25000000,N/A
5,Lavington,Townhouse,5,6,N/A,En Suite,75000000,N/A
6,Lavington,Townhouse,5,6,N/A,En Suite,81000000,N/A
7,Lavington,Townhouse,6,7,N/A,En Suite,85000000,N/A
8,Westlands,Land,2,N/A,N/A,En Suite,28644620,N/A
9,Kileleshwa,Apartment,2,2,N/A,En Suite,9210000,N/A


## 9. Save Output Files

In [19]:
RAW_PATH    = 'data/raw_listings.csv'
SOURCE_PATH = 'data/raw_listings_by_source.csv'
DICT_PATH   = 'data/data_dictionary.csv'

df_listings.to_csv(RAW_PATH, index=False, encoding='utf-8-sig')
print(f'[OK]  {RAW_PATH}  ->  {len(df_listings)} rows  (sprint deliverable)')

df_with_source.to_csv(SOURCE_PATH, index=False, encoding='utf-8-sig')
print(f'[OK]  {SOURCE_PATH}  ->  {len(df_with_source)} rows  (includes Source column)')

[OK]  data/raw_listings.csv  ->  926 rows  (sprint deliverable)
[OK]  data/raw_listings_by_source.csv  ->  926 rows  (includes Source column)


## 10. Generate & Save Data Dictionary

In [20]:
source_names = ', '.join(s['name'] for s in SOURCES)

data_dict_records = [
    {
        'Column Name' : 'Location',
        'Meaning'     : 'Sub-location or neighbourhood within Nairobi',
        'Data Type'   : 'String',
        'Example'     : 'Westlands',
        'Notes'       : 'Matched against 85+ known Nairobi sub-locations (longest-match-first). '
                        'Defaults to "Nairobi" when sub-location cannot be parsed. '
                        'Standardise and group sparse areas in Day 2.',
    },
    {
        'Column Name' : 'Property Type',
        'Meaning'     : 'Category of property inferred from listing title and description',
        'Data Type'   : 'String (categorical)',
        'Example'     : 'Apartment',
        'Notes'       : 'One of: Apartment, House, Townhouse, Villa, Maisonette, Bungalow, '
                        'Studio, Bedsitter, Land, Commercial. Defaults to Apartment.',
    },
    {
        'Column Name' : 'Bedrooms',
        'Meaning'     : 'Number of bedrooms',
        'Data Type'   : 'Integer / N/A',
        'Example'     : '3',
        'Notes'       : 'N/A for Land and some Commercial listings. '
                        'Cast with pd.to_numeric(errors="coerce") before modelling.',
    },
    {
        'Column Name' : 'Bathrooms',
        'Meaning'     : 'Number of bathrooms',
        'Data Type'   : 'Integer / N/A',
        'Example'     : '2',
        'Notes'       : 'N/A if not stated in the listing.',
    },
    {
        'Column Name' : 'Size (m2)',
        'Meaning'     : 'Total floor area or plot size in square metres',
        'Data Type'   : 'Float / N/A',
        'Example'     : '120.0',
        'Notes'       : 'Auto-converts: sqft x 0.0929, acres x 4046.86. N/A if not stated.',
    },
    {
        'Column Name' : 'Amenities',
        'Meaning'     : 'Comma-separated list of detected facilities and features',
        'Data Type'   : 'String (multi-value)',
        'Example'     : 'Swimming Pool, Parking, Security, Gym',
        'Notes'       : '"None" if no keywords detected. 30 amenity types scanned across all sources. '
                        'In Day 2, derive amenity_score = number of amenities detected.',
    },
    {
        'Column Name' : 'Price (KES)',
        'Meaning'     : 'Asking price in Kenyan Shillings (digits only)',
        'Data Type'   : 'Integer / N/A',
        'Example'     : '12500000',
        'Notes'       : 'TARGET VARIABLE. Handles KSh/KES/Kshs prefix, xM shorthand, bare integers. '
                        'N/A if unparseable. Cast with pd.to_numeric(errors="coerce").',
    },
    {
        'Column Name' : 'Listing Date',
        'Meaning'     : 'Date or relative label when the property was listed',
        'Data Type'   : 'String / N/A',
        'Example'     : '15 May 2024',
        'Notes'       : 'Absolute or relative (e.g. "3 days ago"). '
                        'N/A on sources that do not expose listing dates (PropertyPro, KPC, Maploti). '
                        'Normalise in Day 2 to derive the month feature.',
    },
]

df_dict = pd.DataFrame(data_dict_records)
df_dict.to_csv(DICT_PATH, index=False, encoding='utf-8-sig')
print(f'[OK]  {DICT_PATH}  ->  {len(df_dict)} columns documented\n')
df_dict

[OK]  data/data_dictionary.csv  ->  8 columns documented



,Column Name,Meaning,Data Type,Example,Notes
0,Location,Sub-location or neighbourhood within Nairobi,String,Westlands,Matched against 85+ known Nairobi sub-location...
1,Property Type,Category of property inferred from listing tit...,String (categorical),Apartment,"One of: Apartment, House, Townhouse, Villa, Ma..."
2,Bedrooms,Number of bedrooms,Integer / N/A,3,N/A for Land and some Commercial listings. Cas...
3,Bathrooms,Number of bathrooms,Integer / N/A,2,N/A if not stated in the listing.
4,Size (m2),Total floor area or plot size in square metres,Float / N/A,120.0,"Auto-converts: sqft x 0.0929, acres x 4046.86...."
5,Amenities,Comma-separated list of detected facilities an...,String (multi-value),"Swimming Pool, Parking, Security, Gym","""None"" if no keywords detected. 30 amenity typ..."
6,Price (KES),Asking price in Kenyan Shillings (digits only),Integer / N/A,12500000,"TARGET VARIABLE. Handles KSh/KES/Kshs prefix, ..."
7,Listing Date,Date or relative label when the property was l...,String / N/A,15 May 2024,"Absolute or relative (e.g. ""3 days ago""). N/A ..."


## 11. Day 1 Deliverables Checklist

In [21]:
REQUIRED_COLS = ['Location', 'Property Type', 'Bedrooms', 'Bathrooms',
                 'Size (m2)', 'Amenities', 'Price (KES)', 'Listing Date']

checks = [
    (os.path.exists(RAW_PATH),
     f'data/raw_listings.csv             ({len(df_listings)} rows)'),
    (os.path.exists(DICT_PATH),
     f'data/data_dictionary.csv          ({len(df_dict)} columns documented)'),
    (os.path.exists(SOURCE_PATH),
     f'data/raw_listings_by_source.csv   (audit file)'),
    (len(df_listings) >= 300,
     f'Listing count >= 300              ({len(df_listings)} collected)'),
    (all(c in df_listings.columns for c in REQUIRED_COLS),
     'All 8 sprint-required columns present'),
]

print('=' * 65)
print('  DAY 1 DELIVERABLES CHECKLIST')
print('=' * 65)
all_ok = True
for ok, label in checks:
    status = 'OK     ' if ok else 'MISSING'
    print(f'  [{status}]  {label}')
    if not ok:
        all_ok = False

print()
print('  Source breakdown (post-dedup):')
if not df_with_source.empty:
    vc = df_with_source['Source'].value_counts()
    for src, cnt in vc.items():
        bar = chr(9608) * min(cnt // 5, 35)
        print(f'    {src:<30} {cnt:>5}  {bar}')

print()
status_msg = 'ALL CHECKS PASSED' if all_ok else 'SOME CHECKS FAILED - see above'
print(f'  Status: {status_msg}')
print()
print('  Remaining manual step:')
print('    [ ] git init && git add data/ day1_data_collection.ipynb')
print('    [ ] git commit -m "Day 1: raw dataset from 10 sources"')
print('    [ ] git push origin main')
print('=' * 65)

  DAY 1 DELIVERABLES CHECKLIST
  [OK     ]  data/raw_listings.csv             (926 rows)
  [OK     ]  data/data_dictionary.csv          (8 columns documented)
  [OK     ]  data/raw_listings_by_source.csv   (audit file)
  [OK     ]  Listing count >= 300              (926 collected)
  [OK     ]  All 8 sprint-required columns present

  Source breakdown (post-dedup):
    Jiji                             283  ███████████████████████████████████
    PropertyPro                      222  ███████████████████████████████████
    KenyaPropertyCentre              213  ███████████████████████████████████
    BuyRentKenya                     206  ███████████████████████████████████
    CommercialPropertyKenya            2  

  Status: ALL CHECKS PASSED

  Remaining manual step:
    [ ] git init && git add data/ day1_data_collection.ipynb
    [ ] git commit -m "Day 1: raw dataset from 10 sources"
    [ ] git push origin main
